In [28]:
import os
import scanpy as sc
import pandas as pd
import squidpy as sq
import anndata as ad
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [29]:
in_dir = "../../results/09_analysis_podocyte_niche/08_identify_Periglomerular"
out_dir = "../../results/09_analysis_podocyte_niche/13_compare_cell_type"
os.makedirs(out_dir, exist_ok=True)

In [30]:
sample_list = [
    "15837",
    "15852",
    "16512",
    "21022",
    "8262",
    "B23-03794-5",
    "B23-05785-7-1",
    "B23-05785-7-2",
    "X1842",
    "X1845",
    "X1899",
    "X1912",
]

df_list = []
for sample in sample_list:
    adata = sc.read_h5ad(f"{in_dir}/{sample}.h5ad")

    df_list.append(adata.obs[["sample", "annotation_v2"]])

df_anno = pd.concat(df_list, axis=0)
df_anno["barcode"] = df_anno.index
df_anno.reset_index(drop=True, inplace=True)

In [31]:
df_anno

,sample,annotation_v2,barcode
0,sample_15837,Tubulointerstitial,AACCAAGGTATCAGGC-1
1,sample_15837,Tubulointerstitial,AACCGCCAGACTACTT-1
2,sample_15837,Tubulointerstitial,AACCTACCTACCGGTT-1
3,sample_15837,Tubulointerstitial,AACGAATTGACCGGTT-1
4,sample_15837,Tubulointerstitial,AACTCAACCTTGACCA-1
...,...,...,...
19842,sample_X1912,Tubulointerstitial,TGTTCTGCTCTGTCGT-1
19843,sample_X1912,Tubulointerstitial,TGTTGGAACCTTCCGC-1
19844,sample_X1912,Periglomerular,TGTTGGAACGAGGTCA-1
19845,sample_X1912,Tubulointerstitial,TGTTGGATGGACTTCT-1


In [32]:
adata = sc.read_h5ad(f"../../results/04_spatial_clustering/01_run_cellcharter/adata_cellcharter_merged.h5ad")

df_cell_type = adata.obs

In [33]:
df_cell_type["barcode"] = df_cell_type.index.str.split("_").str[0]

In [34]:
df_cell_type

,in_tissue,array_row,array_col,sample,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,...,NEU,PC,PEC,POD,PT,PapE,TAL,VSM_P,cluster_cellcharter,barcode
AACCAAGGTATCAGGC-1_0,1.0,38.0,104.0,sample_15837,7183,8.879612,23835.0,10.078953,16.731697,21.875393,...,0.048070,0.040052,0.369468,0.487072,1.194311,0.106117,0.234746,0.173573,3,AACCAAGGTATCAGGC-1
AACCGCCAGACTACTT-1_0,1.0,39.0,45.0,sample_15837,2865,7.960673,5314.0,8.578288,24.049680,29.318781,...,0.016190,0.072695,0.006890,0.000186,1.303783,0.072608,0.740944,0.043523,1,AACCGCCAGACTACTT-1
AACGAATTGACCGGTT-1_0,1.0,35.0,39.0,sample_15837,3985,8.290544,7695.0,8.948456,16.530214,21.247563,...,0.017627,0.170707,0.014031,0.000280,2.737952,0.025408,0.167583,0.026320,8,AACGAATTGACCGGTT-1
AACTCAACCTTGACCA-1_0,1.0,39.0,79.0,sample_15837,4318,8.370779,8361.0,9.031453,19.973687,23.992345,...,0.059309,0.029975,0.015561,0.000779,0.285656,0.011040,0.004499,0.097398,1,AACTCAACCTTGACCA-1
AACTGCCTCGATAGGT-1_0,1.0,37.0,61.0,sample_15837,5434,8.600615,10826.0,9.289799,12.848698,16.709773,...,0.027911,0.316204,0.040908,0.001074,1.441686,0.082971,0.277265,0.073773,1,AACTGCCTCGATAGGT-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TGTTCTGCTCTGTCGT-1_11,1.0,39.0,39.0,sample_X1912,11579,9.357035,75943.0,11.237752,12.051144,15.844778,...,0.024937,0.007707,0.096716,0.015927,12.942639,0.000367,7.341363,0.023508,12,TGTTCTGCTCTGTCGT-1
TGTTGGAACCTTCCGC-1_11,1.0,35.0,35.0,sample_X1912,12109,9.401787,109936.0,11.607663,10.142265,14.271030,...,0.067327,0.012484,4.455663,0.001088,9.092853,0.275564,0.003863,0.118828,10,TGTTGGAACCTTCCGC-1
TGTTGGAACGAGGTCA-1_11,1.0,28.0,72.0,sample_X1912,12692,9.448806,105787.0,11.569192,13.993213,18.038133,...,0.308677,0.026867,5.803476,0.609641,1.112008,0.054332,0.001189,0.732970,10,TGTTGGAACGAGGTCA-1
TGTTGGATGGACTTCT-1_11,1.0,13.0,53.0,sample_X1912,11018,9.307376,62965.0,11.050350,9.659335,13.483681,...,0.047232,0.002001,0.083439,0.063336,19.579206,0.010703,0.012048,0.796745,11,TGTTGGATGGACTTCT-1


In [35]:
df = pd.merge(df_cell_type, 
              df_anno, 
              on=["sample", "barcode"], how="left")

df = df[["sample", "annotation_v2",'ATL', 'CNT', 'DCT', 'DTL',
       'EC', 'FIB', 'IC', 'IMM_B', 'IMM_MAC_M2', 'IMM_MAST', 'IMM_MDC',
       'IMM_N', 'IMM_NKT', 'IMM_PL', 'IMM_T', 'IMM_cDC', 'IMM_cycMNP',
       'IMM_ncMON', 'IMM_pDC', 'NEU', 'PC', 'PEC', 'POD', 'PT', 'PapE', 'TAL',
       'VSM_P']]

# compute mean cell type proportions for each sample and annotation_v2 group
df = df.groupby(['sample', 'annotation_v2']).mean().reset_index()

In [36]:
# convert to proportion by dividing each row by the sum of the row
# except for the first two columns
df.iloc[:, 2:] = df.iloc[:, 2:].div(df.iloc[:, 2:].sum(axis=1), axis=0)

In [37]:
# add group info to adata.obs based on sample column in adata.obs
group_dict = {
    'sample_15837': 'SRC',
    'sample_15852': 'SRC',
    'sample_16512': 'SRC',
    'sample_21022': 'SRC',
    'sample_8262': 'SRC',
    'sample_B23-03794-5': 'Normal',
    'sample_B23-05785-7-1': 'Normal',
    'sample_B23-05785-7-2': 'Normal',
    'sample_X1842': 'LN',
    'sample_X1845': 'LN',
    'sample_X1899': 'LN',
    'sample_X1912': 'LN'
}
df['group'] = df['sample'].map(group_dict)

In [38]:
df.head()

,sample,annotation_v2,ATL,CNT,DCT,DTL,EC,FIB,IC,IMM_B,...,IMM_pDC,NEU,PC,PEC,POD,PT,PapE,TAL,VSM_P,group
0,sample_15837,Glomerular,0.015812,0.006899,0.012066,0.016149,0.078625,0.088254,0.019846,0.013188,...,0.014577,0.012385,0.040960,0.033834,0.077658,0.331580,0.009311,0.051956,0.030717,SRC
1,sample_15837,Periglomerular,0.018206,0.023039,0.016884,0.022534,0.070993,0.088630,0.044496,0.015898,...,0.014276,0.011534,0.027750,0.015754,0.034747,0.287942,0.011658,0.097402,0.042962,SRC
2,sample_15837,Tubulointerstitial,0.021995,0.033876,0.026569,0.020872,0.069833,0.091225,0.058898,0.013859,...,0.016252,0.011190,0.032539,0.017287,0.022249,0.229120,0.013053,0.111346,0.045930,SRC
3,sample_15852,Glomerular,0.003297,0.004003,0.003078,0.014825,0.131205,0.055426,0.005626,0.015197,...,0.043856,0.051808,0.006422,0.036740,0.197564,0.096847,0.004363,0.014397,0.126000,SRC
4,sample_15852,Periglomerular,0.007883,0.011712,0.013567,0.013977,0.067341,0.081433,0.024432,0.013213,...,0.038987,0.042736,0.014643,0.018237,0.034688,0.332954,0.005274,0.024335,0.082880,SRC


In [39]:
df.to_csv(f"{out_dir}/proportion_cell_type_with_sample.csv", 
          index=False)
df.drop(columns='sample', inplace=True)

In [40]:
df = df.groupby(['group', 'annotation_v2']).mean().reset_index()

In [41]:
# # convert to proportion by dividing each row by the sum of the row
# # except for the first two columns
# df.iloc[:, 2:] = df.iloc[:, 2:].div(df.iloc[:, 2:].sum(axis=1), axis=0)

In [42]:
# plot stacked barplot of df
for anno in df['annotation_v2'].unique():
    df_anno = df[df['annotation_v2'] == anno]

    df_anno.set_index('group', inplace=True)
    df_anno = df_anno.sort_values(by='group')

    fig, ax = plt.subplots(figsize=(10, 6))

    # add white lines between bars
    df_anno.plot(kind='bar', stacked=True, ax=ax,
                 figsize=(10, 6), colormap='tab20', edgecolor='white')
    plt.ylabel('Proportion')
    plt.xlabel('Group')
    plt.title(f'Proportion of Cell type in Each Group - {anno}')
    plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()

    plt.savefig(f"{out_dir}/proportion_cell_type_{anno}.png", dpi=300)
    plt.close()

In [43]:
df

,group,annotation_v2,ATL,CNT,DCT,DTL,EC,FIB,IC,IMM_B,...,IMM_ncMON,IMM_pDC,NEU,PC,PEC,POD,PT,PapE,TAL,VSM_P
0,LN,Glomerular,0.000803,0.011626,0.011369,0.003046,0.129637,0.073729,0.007881,0.010036,...,0.033969,0.003324,0.003042,0.003669,0.148618,0.114246,0.161353,0.000708,0.017109,0.152395
1,LN,Periglomerular,0.001458,0.032166,0.021414,0.006234,0.046874,0.097099,0.020579,0.015395,...,0.013607,0.003308,0.003445,0.005235,0.067488,0.019663,0.420188,0.000984,0.041665,0.049896
2,LN,Tubulointerstitial,0.002422,0.025602,0.015039,0.009609,0.029971,0.088732,0.014960,0.010814,...,0.006750,0.002382,0.003128,0.006455,0.019320,0.002771,0.511356,0.001299,0.108355,0.039902
3,Normal,Glomerular,0.002453,0.030073,0.036026,0.007074,0.155251,0.025349,0.029275,0.008520,...,0.024180,0.007447,0.005454,0.016882,0.033014,0.208611,0.185102,0.003504,0.042111,0.098904
4,Normal,Periglomerular,0.004682,0.051839,0.070862,0.007620,0.068025,0.032043,0.054394,0.008607,...,0.017672,0.007951,0.008656,0.024180,0.017266,0.066674,0.346644,0.004176,0.056451,0.064871
5,Normal,Tubulointerstitial,0.007486,0.039611,0.051383,0.038500,0.037489,0.056828,0.045627,0.008280,...,0.013035,0.008086,0.009437,0.076637,0.007568,0.011287,0.214096,0.004455,0.239861,0.042937
6,SRC,Glomerular,0.007966,0.007842,0.011357,0.010796,0.132912,0.066158,0.016660,0.013328,...,0.023046,0.024651,0.020829,0.013093,0.039164,0.201476,0.158389,0.010252,0.022247,0.080977
7,SRC,Periglomerular,0.012681,0.021766,0.022614,0.014070,0.075730,0.100658,0.036654,0.013321,...,0.020934,0.022275,0.021111,0.018519,0.017978,0.034051,0.298020,0.013236,0.038020,0.063841
8,SRC,Tubulointerstitial,0.016311,0.020560,0.020896,0.022664,0.060951,0.108105,0.033555,0.011526,...,0.018676,0.020647,0.019620,0.032002,0.011342,0.007892,0.302205,0.013031,0.091197,0.045019


In [44]:
df.to_csv(f"{out_dir}/proportion_cell_type.csv", index=False)